In [8]:
from qiskit import QuantumCircuit, QuantumRegister
#from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator
import torch
#import torch.nn.functional as F
import numpy as np
import math
from common import SoftThresholding, find_min_power
from IPython.display import display, Math
import sympy as sp
from wt_functions import *

np.set_printoptions(suppress=True, precision=8)

In [9]:
def haar_transform_2d_qiskit(x, inverse=False, noise=0.01):
    # assert checks for batch and channel = 1
    B, C, H, W = x.shape
    assert B == 1 and C == 1, "function assumes B and C are = to 1"
    assert noise <= 1 and noise >= 0, "noise range: [0,1]"

    # pad h and w to the next power of 2
    H_pad_size = next_power_of_2(H)
    W_pad_size = next_power_of_2(W)
    X = np.zeros((H_pad_size, W_pad_size), dtype=float)
    X[:H, :W] = x[0, 0] # new padded image

    psi = X.reshape(-1) # flatten
    norm = np.linalg.norm(psi)
    if norm == 0:
        raise ValueError("Input is all zeros; cannot normalize for amplitude encoding.")
    amps = psi / norm # flatten normalized list of probability amplitudes

    # find num qubits
    n_row = int(math.log2(H_pad_size))
    n_col = int(math.log2(W_pad_size))
    n_qubits = n_row + n_col

    # qubit numbers
    col_qubits = list(range(n_col))
    row_qubits = list(range(n_col, n_col+n_row))

    # haar matrices inverse if true
    transform_H = haar_matrix_builder(H_pad_size)
    transform_W = haar_matrix_builder(W_pad_size)
    if inverse:
        transform_H = transform_H.T
        transform_W = transform_W.T
        
    # turn haar transform into operators
    U_row = Operator(transform_H)
    U_col = Operator(transform_W)

    # create q-circuit and encode amplitudes
    qc = QuantumCircuit(n_qubits)
    qc.initialize(amps, list(range(n_qubits)))

    # put the haar transforms into the circuit
    qc.append(U_col, col_qubits)
    qc.append(U_row, row_qubits)

    # extract amplitudes after applying haar
    sv_ideal = Statevector.from_instruction(qc).data # ideal
    Yq_ideal = sv_ideal.reshape(H_pad_size, W_pad_size)
    Yq_mean, Yq_std = haar_noise(sv_ideal, H_pad_size, W_pad_size, n_qubits, p=noise, trials=500)
    
    # get the normalized images back to shape and compare with classical implementation
    Yc = haar_transform_2d_classical(X) / norm

    return Yq_ideal, Yq_mean, Yq_std, Yc, qc

In [10]:
if __name__ == "__main__":
    # example here is 32x32
    #np.random.seed(0)
    X = np.random.rand(1, 1, 8, 8).astype(float)

    # noise is from 0 to 1     1 being most noise
    Yq_ideal, Yq_mean, Yq_std, Yc, qc = haar_transform_2d_qiskit(X, inverse=False, noise=0.01)

    # error between quantum and classical implementation of haar transform
    err_ideal = np.max(np.abs(np.real(Yq_ideal) - Yc))
    err_noise = np.max(np.abs(np.real(Yq_ideal) - Yq_mean))
    # print("max ideal error:", err_ideal)
    # print("max noisy error:", err_noise)

    # print matrix patch or full
    n_patch = 4
    # print("\nYq (ideal)\n", np.real(Yq_ideal[:n_patch, :n_patch]))
    # print("\nYq (mean with noise)\n", np.real(Yq_mean[:n_patch, :n_patch]))
    # print("Sensitivity to noise:\n", Yq_std[:n_patch, :n_patch]) # sensitivity eachs matrix element is to noise
    # print("\nYc\n", Yc[:n_patch, :n_patch])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>